In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os 
from thefuzz import process
import pandas as pd
import os
import re
import holidays
import tkinter as tk
from tkinter import filedialog
from collections import defaultdict
from clase_reportes_new import ReportClassNew
import json
rc = ReportClassNew()

# Paises

In [2]:
ruta = rc.validar_ruta()

In [3]:
ruta_ecuador = ruta / 'data' / 'paises' / 'REPÚBLICA DOMINICANA'

In [4]:
ruta_ecuador

WindowsPath('G:/Otros ordenadores/Mi portátil/VENTA MENSUAL/data/paises/REPÚBLICA DOMINICANA')

In [5]:
archivo1 = pd.read_excel(ruta_ecuador /  'SELL IN 2025.xlsx')

In [6]:
# Encontrar el índice de la fila que contiene "Total" en la primera columna
index_total = int(archivo1[archivo1.iloc[:, 0:1].iloc[:, 0] == 'Total'].index[0])
archivo1.columns =     archivo1[index_total-2:index_total-1].values[0]
index_fechas = [i for i in range(len(archivo1.columns)) if pd.notna(archivo1.columns[i])]
index_fechas.insert(0, 0)

archivo1= archivo1[index_total+1:].iloc[:, index_fechas]
meses = {'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4, 'mayo': 5, 'junio': 6,
         'julio': 7, 'agosto': 8, 'septiembre': 9, 'octubre': 10, 'noviembre': 11, 'diciembre': 12}
archivo1.rename(columns={archivo1.columns[0]: 'Detalles'}, inplace=True)
archivo1['Clientes'] = np.where(
    archivo1['Detalles'].str.contains('[', regex=False, na=False),
    np.nan,
    archivo1['Detalles']
)
archivo1['Clientes'] = archivo1['Clientes'].ffill()
archivo1 = archivo1[archivo1['Detalles'].str.contains('[', regex=False, na=False)]
archivo1 = archivo1.melt(id_vars=['Clientes', 'Detalles'], var_name='Fecha', value_name='Valor')
archivo1['Fecha'] = [
    pd.to_datetime(f'{y}-{meses[mes]}-01')
    for mes, y in (i.split(' ') for i in archivo1['Fecha'])
]
archivo1['Valor'] = pd.to_numeric(archivo1['Valor'], errors='coerce')
archivo1 = archivo1[archivo1['Valor']>0 ]

In [ ]:
archivo1['Sucursal'] = archivo1['Clientes'].str.split(',').str[1]

archivo1.loc[
    archivo1['Clientes']
    .str.split('-').str[1]
    .str.lower()
    .str.contains(r'pocion|\.do|whats', na=False),
    'Segmento'
] = 'Ecommerce'
archivo1.loc[~archivo1['Clientes'].str.contains(',|-'), 'Segmento'] = 'Mayorista'
archivo1.loc[archivo1['Clientes'].str.lower().str.contains('byg|presencial'), 'Segmento'] = 'BYG'
archivo1['Clientes'] = archivo1['Clientes'].str.split(',').str[0]
archivo1['Clientes'] = archivo1['Clientes'].str.strip().str.replace(r'\s+', ' ', regex=True)


In [ ]:

archivo1.to_excel(r'D:\Desktop\ventas_dominicana.xlsx', index=False)

In [21]:
## Procesar archivos de República Dominicana
import os
import pandas as pd
import numpy as np

carpeta_seleccionada = r'g:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\paises\REPÚBLICA DOMINICANA'
lista_dataframes = []

for archivo in os.listdir(carpeta_seleccionada):
    if archivo.endswith(('.xls', '.xlsx')):   # acepta ambos tipos
        ruta_completa = os.path.join(carpeta_seleccionada, archivo)
        try:
            archivo1 = pd.read_excel(ruta_completa, header=None)
            # Encontrar el índice de la fila que contiene "Total" en la primera columna
            index_total = int(archivo1[archivo1.iloc[:, 0:1].iloc[:, 0] == 'Total'].index[0])
            archivo1.columns =     archivo1[index_total-2:index_total-1].values[0]
            index_fechas = [i for i in range(len(archivo1.columns)) if pd.notna(archivo1.columns[i])]
            index_fechas.insert(0, 0)

            archivo1= archivo1[index_total+1:].iloc[:, index_fechas]
            meses = {'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4, 'mayo': 5, 'junio': 6,
                    'julio': 7, 'agosto': 8, 'septiembre': 9, 'octubre': 10, 'noviembre': 11, 'diciembre': 12}
            archivo1.rename(columns={archivo1.columns[0]: 'Detalles'}, inplace=True)
            archivo1['Clientes'] = np.where(
                archivo1['Detalles'].str.contains('[', regex=False, na=False),
                np.nan,
                archivo1['Detalles']
            )
            archivo1['Clientes'] = archivo1['Clientes'].ffill()
            archivo1 = archivo1[archivo1['Detalles'].str.contains('[', regex=False, na=False)]
            archivo1 = archivo1.melt(id_vars=['Clientes', 'Detalles'], var_name='Fecha', value_name='Valor')
            archivo1['Fecha'] = [
                pd.to_datetime(f'{y}-{meses[mes]}-01')
                for mes, y in (i.split(' ') for i in archivo1['Fecha'])
            ]
            archivo1['Valor'] = pd.to_numeric(archivo1['Valor'], errors='coerce')
            archivo1 = archivo1[archivo1['Valor']>0 ]
            archivo1['Sucursal'] = archivo1['Clientes'].str.split(',').str[1]
            archivo1.loc[
                archivo1['Clientes']
                .str.split('-').str[1]
                .str.lower()
                .str.contains(r'pocion|\.do|whats', na=False),
                'Segmento'
            ] = 'Ecommerce'
            archivo1.loc[~archivo1['Clientes'].str.contains(',|-'), 'Segmento'] = 'Mayorista'
            archivo1.loc[archivo1['Clientes'].str.lower().str.contains('byg|presencial'), 'Segmento'] = 'BYG'
            archivo1['Clientes'] = archivo1['Clientes'].str.split(',').str[0]
            archivo1['Detalles'] = archivo1['Detalles'].str.strip().str.replace(r'\s+', ' ', regex=True)
            archivo1['Clientes'] = archivo1['Clientes'].str.strip().str.replace(r'\s+', ' ', regex=True)
            archivo1['Valor'] = archivo1['Valor'].astype(int)
            lista_dataframes.append(archivo1)
        except Exception as e:
            print(f"No se pudo leer el archivo '{archivo}'. Error: {e}")



print("Concatenando todos los archivos...")
if lista_dataframes:

    df_concatenado = pd.concat(lista_dataframes, ignore_index=True, )
    df_concatenado = df_concatenado.loc[:, ~df_concatenado.columns.duplicated()]
    df_concatenado = df_concatenado[df_concatenado['Clientes'].notna()]
    print(f"¡Consolidación completada! Total filas: {len(df_concatenado)}")


Concatenando todos los archivos...
¡Consolidación completada! Total filas: 2773


In [2]:
df_concatenado.to_excel(r'D:\Desktop\ventas_dominicana.xlsx', index=False)

In [7]:
lista_dataframes[2]


,Clientes,Detalles,Fecha,Valor
47,Ana Rita Ferreira -BYG,[PCN04] LA POCION MASCARILLA ANCESTR...,2025-01-01,1.0
53,Angely Vega-BYG,[TNGL01] LA POCION SHAMPOO TONGOLE 4...,2025-01-01,1.0
54,Angely Vega-BYG,[TNGL02] LA POCION TRATAMIENTO TONGO...,2025-01-01,1.0
55,Angely Vega-BYG,[TNGL03] LA POCION LEAVE ON TONGOLE ...,2025-01-01,1.0
69,BLUE WI SDQ SRL (ALISS DOMINICANA),[PCN07] LA POCION OLEO CAPILAR LINEA B8,2025-01-01,3.0
...,...,...,...,...
23560,vilexy de la cruz-pocion.do,[TNGL03] LA POCION LEAVE ON TONGOLE ...,2025-12-01,1.0
23561,vilexy de la cruz-pocion.do,[TNGL04] LA POCION MASCARILLA TONGOL...,2025-12-01,1.0
23562,vilexy de la cruz-pocion.do,[TNGL05] LA POCION GEL DEFINIDOR TON...,2025-12-01,1.0
23622,yesmailin guzman-pocion.do,[PCN19] LA POCION DUTONIC CONTROL CA...,2025-12-01,1.0


In [ ]:
# ## Procesar archivos de Perú
# import os
# import pandas as pd

# carpeta_seleccionada = r'g:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\paises\PERÚ'
# lista_dataframes = []

# for archivo in os.listdir(carpeta_seleccionada):
#     if archivo.endswith(('.xls', '.xlsx')):   # acepta ambos tipos
#         ruta_completa = os.path.join(carpeta_seleccionada, archivo)
#         try:
#             df_peru = pd.read_excel(ruta_completa)
#             df_peru = (
#                 df_peru
#                 .melt(id_vars='SKU', var_name='Fecha', value_name='Ventas')
#                 .fillna({'Ventas': 0})
#             )
#             df_peru['Fecha'] = pd.to_datetime(df_peru['Fecha'], errors='coerce')
#             lista_dataframes.append(df_peru)
#         except Exception as e:
#             print(f"No se pudo leer el archivo '{archivo}'. Error: {e}")



# print("Concatenando todos los archivos...")
# if lista_dataframes:

#     df_concatenado = pd.concat(lista_dataframes, ignore_index=False, axis=1)
#     df_concatenado = df_concatenado.loc[:, ~df_concatenado.columns.duplicated()]
#     print(f"¡Consolidación completada! Total filas: {len(df_concatenado)}")


Concatenando todos los archivos...
¡Consolidación completada! Total filas: 154
